# Leafcheck: Maize Leaf Disease Classifier & Non-Maize Rejection Training (Google Colab)

This notebook trains a 5-class **MobileNetV2** model designed to:
1. **Distinguish Maize vs. Non-Maize** (`Not_Leaf`):
   - **Sugarcane & Grassy Monocot Leaves**: Real sugarcane leaf disease images from an open GitHub mirror (zero authentication required) + procedural parallel-veined monocot grass samples.
   - **Other Crop Leaves**: Tomato, potato, apple, grape, pepper, etc. (PlantVillage).
   - **Flowers & Blooms**: Petals, roses, sunflowers, orchids (Oxford Flowers 102).
   - **Written Text, Documents & Handwriting**: Invoices, handwriting, notebook paper, math notes, code.
   - **Everyday Objects & Scenes**: Cars, animals, indoor scenes, furniture (Imagenette).
2. **Accurately Classify Maize Foliar Diseases**:
   - **Northern Leaf Blight** (*Exserohilum turcicum*)
   - **Common Rust** (*Puccinia sorghi*)
   - **Gray Leaf Spot** (*Cercospora zeae-maydis*)
   - **Healthy Maize**
3. **Compute Mahalanobis Out-of-Distribution Gating** (`ood_stats.npz`) to reject unfamiliar inputs.

### Instructions:
1. Set runtime to **GPU**: Click **Runtime > Change runtime type > T4 GPU > Save**.
2. Click **Runtime > Run all** (or execute each cell sequentially).

In [ ]:
# 1) GPU check and dependency installation
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))
!pip -q install kagglehub scikit-learn matplotlib tensorflow-datasets pillow

## 2) Write Training Engine (`train.py` & `fit_ood.py`)

In [ ]:
%%writefile train.py
#!/usr/bin/env python3
"""Train the corn leaf classifier (MobileNetV2 transfer learning) with robust non-maize rejection."""
import argparse
import csv
import hashlib
import io
import json
import os
import random
import re
import shutil
import sys
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import keras  # noqa: E402
current_script_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
sys.path.insert(0, str(current_script_dir))
import fit_ood  # noqa: E402
import tensorflow as tf  # noqa: E402
from keras import layers  # noqa: E402
from sklearn.metrics import classification_report, confusion_matrix, f1_score  # noqa: E402
from sklearn.model_selection import train_test_split  # noqa: E402
from sklearn.utils.class_weight import compute_class_weight  # noqa: E402

IMG = 224
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif"}
CACHE_MAX_SIDE = 512
NEG_CLASS = "Not_Leaf"


def canon(folder: str, overrides: dict[str, str] = None) -> str | None:
    if overrides and folder in overrides:
        return overrides[folder]
    k = re.sub(r"[^a-z0-9]+", "_", folder.lower()).strip("_")
    tokens = set(k.split("_"))

    if tokens & {"not", "non", "other", "others", "negative", "background", "not_corn", "not_leaf", "non_corn", "non_maize"}:
        return NEG_CLASS

    # 1. Sugarcane, sorghum, and grassy monocot leaves
    monocots = {"sugarcane", "sugar_cane", "saccharum", "cane", "sorghum", "bamboo", "rice", "paddy", "wheat", "grass", "weed", "weeds", "poaceae"}
    if tokens & monocots:
        return NEG_CLASS

    # 2. Other plant species
    non_corn = {"apple", "banana", "blueberry", "cassava", "cherry", "citrus", "coffee", "cotton", "grape", "mango", "olive", "onion", "orange", "peach", "pepper", "potato", "raspberry", "soybean", "squash", "strawberry", "tea", "tomato"}
    if tokens & non_corn:
        return NEG_CLASS

    # 3. Flowers & blooms
    flowers = {"flower", "flowers", "rose", "orchid", "sunflower", "tulip", "petal", "bloom", "hibiscus", "dahlia"}
    if tokens & flowers:
        return NEG_CLASS

    # 4. Text, writings, documents
    docs = {"text", "document", "documents", "doc", "paper", "writing", "handwriting", "book", "invoice", "receipt", "notes", "code", "drawing"}
    if tokens & docs:
        return NEG_CLASS

    # 5. Non-plant objects and scenes
    objects = {"object", "objects", "imagenet", "imagenette", "cifar", "coco", "car", "vehicle", "animal", "indoor", "room", "synthetic"}
    if tokens & objects:
        return NEG_CLASS

    # Corn conditions
    if "rust" in k:
        return "Common_Rust"
    if "blight" in k:
        return "Blight"
    if tokens & {"gray", "grey", "cercospora", "spot"}:
        return "Gray_Leaf_Spot"
    if "healthy" in k and ("corn" in k or "maize" in k or len(tokens) == 1):
        return "Healthy"

    return None


def find_images(roots, overrides, skip=()):
    items, unknown = [], Counter()
    for root in roots:
        for d in sorted(Path(root).rglob("*")):
            if not d.is_dir() or d.name in skip:
                continue
            files = [f for f in d.iterdir() if f.is_file() and f.suffix.lower() in EXTS]
            if not files:
                continue
            cls = canon(d.name, overrides)
            if cls is None:
                unknown[str(d)] += len(files)
                continue
            items += [(f, cls) for f in files]
    if unknown:
        for p, _ in unknown.items():
            d = Path(p)
            files = [f for f in d.iterdir() if f.is_file() and f.suffix.lower() in EXTS]
            items += [(f, NEG_CLASS) for f in files]
    return items


def dhash(img: Image.Image) -> int:
    g = np.asarray(img.convert("L").resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
    bits = (g[:, 1:] > g[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


def process(args):
    src, cls, cache = args
    try:
        raw = Path(src).read_bytes()
        img = Image.open(io.BytesIO(raw))
        img.load()
        img = ImageOps.exif_transpose(img).convert("RGB")
        if min(img.size) < 32:
            return None
        md5 = hashlib.md5(raw).hexdigest()
        h = dhash(img)
        img.thumbnail((CACHE_MAX_SIDE, CACHE_MAX_SIDE), Image.Resampling.BILINEAR)
        out = Path(cache) / cls / f"{md5}.jpg"
        out.parent.mkdir(parents=True, exist_ok=True)
        if not out.exists():
            img.save(out, "JPEG", quality=95)
        return dict(path=str(out), label=cls, md5=md5, dhash=h, src=str(src))
    except Exception:
        return None


def clean(items, cache, workers):
    with ThreadPoolExecutor(workers) as ex:
        rows = list(ex.map(process, [(s, c, cache) for s, c in items]))
    ok = [r for r in rows if r]
    report = {"found": len(items), "unreadable_or_tiny": len(items) - len(ok)}

    def dedupe(rows_in, key):
        groups = defaultdict(list)
        for r in rows_in:
            groups[r[key]].append(r)
        kept, dupes, conflicts = [], 0, 0
        for g in groups.values():
            if len({r["label"] for r in g}) > 1:
                conflicts += len(g)
                continue
            kept.append(g[0])
            dupes += len(g) - 1
        return kept, dupes, conflicts

    uniq, exact, c1 = dedupe(ok, "md5")
    final, visual, c2 = dedupe(uniq, "dhash")
    report["exact_duplicates_removed"] = exact
    report["visual_duplicates_removed"] = visual
    report["label_conflicts_removed"] = c1 + c2
    report["kept"] = len(final)
    return final, report


def make_ds(paths, y, batch, training, seed):
    ds = tf.data.Dataset.from_tensor_slices((paths, y))
    if training:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)

    def load(p, label):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(img, (IMG, IMG))
        img = tf.cast(img, tf.float32)  # RAW 0-255 pixels
        img.set_shape((IMG, IMG, 3))
        return img, label

    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch)
    if training:
        aug = keras.Sequential([
            layers.RandomFlip("horizontal_and_vertical"),
            layers.RandomRotation(0.12),
            layers.RandomZoom(0.15),
            layers.RandomContrast(0.2),
            layers.RandomBrightness(0.2, value_range=(0, 255)),
        ])
        ds = ds.map(lambda x, l: (aug(x, training=True), l), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


def build_model(n_classes, pretrained, dropout):
    inp = keras.Input((IMG, IMG, 3), name="input_image")
    x = layers.Rescaling(1 / 127.5, offset=-1, name="rescaling")(inp)
    base = keras.applications.MobileNetV2(
        input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet" if pretrained else None)
    base._name = "mobilenetv2_1.00_224"
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling2d")(x)
    x = layers.Dropout(dropout, name="dropout")(x)
    out = layers.Dense(n_classes, activation="softmax", name="dense")(x)
    return keras.Model(inp, out), base


def fit(model, train, val, lr, epochs, cw, patience):
    model.compile(keras.optimizers.Adam(lr), "sparse_categorical_crossentropy", metrics=["accuracy"])
    cbs = [
        keras.callbacks.EarlyStopping("val_loss", patience=patience, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=max(1, patience // 2), min_lr=1e-7),
    ]
    h = model.fit(train, validation_data=val, epochs=epochs, class_weight=cw, callbacks=cbs, verbose=2)
    return {k: [float(v) for v in vs] for k, vs in h.history.items()}


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--data", nargs="+", required=True, help="dataset root folder(s)")
    ap.add_argument("--out", default="output")
    ap.add_argument("--map", action="append", default=[], metavar="FOLDER=CLASS")
    ap.add_argument("--skip", action="append", default=[])
    ap.add_argument("--max-per-class", type=int, default=1500)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--epochs1", type=int, default=10)
    ap.add_argument("--epochs2", type=int, default=15)
    ap.add_argument("--lr1", type=float, default=1e-3)
    ap.add_argument("--lr2", type=float, default=1e-5)
    ap.add_argument("--unfreeze", type=int, default=40)
    ap.add_argument("--dropout", type=float, default=0.25)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--workers", type=int, default=8)
    a = ap.parse_args()

    random.seed(a.seed); np.random.seed(a.seed); tf.random.set_seed(a.seed)
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    overrides = dict(m.split("=", 1) for m in a.map)

    items = find_images(a.data, overrides, set(a.skip))
    rows, report = clean(items, out / "clean", a.workers)
    if a.max_per_class:
        rng = random.Random(a.seed); rng.shuffle(rows)
        kept, n = [], Counter()
        for r in rows:
            if n[r["label"]] < a.max_per_class:
                n[r["label"]] += 1; kept.append(r)
        rows = kept
    class_names = sorted({r["label"] for r in rows})
    counts = Counter(r["label"] for r in rows)
    print("Cleaning Report:", json.dumps(report, indent=2))
    print("Class Counts:", dict(counts))

    if len(class_names) < 2:
        sys.exit(f"Error: Need at least 2 classes to train. Found: {class_names}")
    if NEG_CLASS not in class_names:
        print("WARNING: No Not_Leaf class found; the model will not be able to reject non-maize!")

    idx = {c: i for i, c in enumerate(class_names)}
    paths = np.array([r["path"] for r in rows]); y = np.array([idx[r["label"]] for r in rows])

    tr_p, tmp_p, tr_y, tmp_y = train_test_split(paths, y, test_size=0.30, stratify=y, random_state=a.seed)
    va_p, te_p, va_y, te_y = train_test_split(tmp_p, tmp_y, test_size=0.50, stratify=tmp_y, random_state=a.seed)

    with open(out / "splits.csv", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["path", "label", "split"])
        for name, ps, ys in (("train", tr_p, tr_y), ("val", va_p, va_y), ("test", te_p, te_y)):
            w.writerows([[p, class_names[i], name] for p, i in zip(ps, ys)])
    print(f"Splits: Train={len(tr_p)}, Val={len(va_p)}, Test={len(te_p)}")

    train = make_ds(tr_p, tr_y, a.batch, True, a.seed)
    val = make_ds(va_p, va_y, a.batch, False, a.seed)
    test = make_ds(te_p, te_y, a.batch, False, a.seed)
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.arange(len(class_names)), y=tr_y)))

    model, base = build_model(len(class_names), True, a.dropout)
    base.trainable = False
    print("\n== Stage 1: Training Classification Head ==")
    h1 = fit(model, train, val, a.lr1, a.epochs1, cw, patience=4)

    base.trainable = True
    for layer in base.layers[:-a.unfreeze]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    print(f"\n== Stage 2: Fine-Tuning Top {a.unfreeze} Backbone Layers ==")
    h2 = fit(model, train, val, a.lr2, a.epochs2, cw, patience=5)

    prob = model.predict(test, verbose=0)
    pred = prob.argmax(1)
    rep = classification_report(te_y, pred, target_names=class_names, output_dict=True, zero_division=0)
    cm = confusion_matrix(te_y, pred)

    model_path = out / "corn_leaf_disease_model_v3.keras"
    model.save(model_path)
    (out / "class_names.json").write_text(json.dumps(class_names, indent=4))

    names = lambda yy: [class_names[i] for i in yy]
    ood = fit_ood.fit_from_splits(
        model, {"train": (list(tr_p), names(tr_y)), "val": (list(va_p), names(va_y)), "test": (list(te_p), names(te_y))},
        out / "ood_stats.npz")
    fit_ood.print_report(ood)

    metrics = {
        "ood_check": ood,
        "classes": class_names, "cleaning": report, "class_counts": dict(counts),
        "split_sizes": {"train": len(tr_p), "val": len(va_p), "test": len(te_p)},
        "test_accuracy": float((pred == te_y).mean()),
        "test_macro_f1": float(f1_score(te_y, pred, average="macro")),
        "per_class": {c: rep[c] for c in class_names},
        "confusion_matrix": cm.tolist(),
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))

    try:
        import matplotlib; matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6, 5)); ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(len(class_names)), class_names, rotation=30, ha="right"); ax.set_yticks(range(len(class_names)), class_names)
        for i in range(len(cm)):
            for j in range(len(cm)):
                ax.text(j, i, cm[i, j], ha="center", va="center", color="black" if cm[i, j] < cm.max() / 2 else "white")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion Matrix"); fig.tight_layout(); fig.savefig(out / "confusion_matrix.png", dpi=150)
    except Exception as e:
        print("Plotting skipped:", e)

    print(f"\nFINAL TEST ACCURACY: {metrics['test_accuracy'] * 100:.2f}%")
    print(classification_report(te_y, pred, target_names=class_names, zero_division=0))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile fit_ood.py
#!/usr/bin/env python3
"""Fit Mahalanobis Out-of-Distribution gate (ood_stats.npz) calibrated on Maize and Non-Maize."""
import json
import os
import sys
from pathlib import Path

import numpy as np
from PIL import Image

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import keras  # noqa: E402

IMG = 224
NEG = "Not_Leaf"


def extract(model, paths, batch=64):
    ext = keras.Model(model.input, model.get_layer("global_average_pooling2d").output)
    out = []
    for i in range(0, len(paths), batch):
        arr = np.stack([np.asarray(Image.open(p).convert("RGB").resize((IMG, IMG), Image.Resampling.BILINEAR), dtype="float32")
                        for p in paths[i:i + batch]])
        out.append(ext.predict(arr, verbose=0))
    return np.concatenate(out).astype(np.float64)


def distances(f, means, precision):
    best = None
    for mu in means:
        d = f - mu
        v = np.einsum("nd,nd->n", d @ precision, d)
        best = v if best is None else np.minimum(best, v)
    return best


def fit_gate(f_train, y_train, shrinkage=0.1):
    classes = sorted({str(c) for c in y_train})
    y = np.asarray(y_train)
    means = np.stack([f_train[y == c].mean(0) for c in classes])
    centred = f_train - means[[classes.index(c) for c in y_train]]
    cov = centred.T @ centred / max(len(f_train) - 1, 1)
    d = cov.shape[0]
    cov = (1 - shrinkage) * cov + shrinkage * np.trace(cov) / d * np.eye(d)
    return classes, means, np.linalg.inv(cov)


def fit_from_splits(model, splits, out_path, percentile=98.5):
    def feats(key, positive):
        paths, names = splits[key]
        keep = [i for i, n in enumerate(names) if (n != NEG) == positive]
        return (extract(model, [paths[i] for i in keep]) if keep else np.zeros((0, 1280))), [names[i] for i in keep]
    f_tr, y_tr = feats("train", True)
    f_va, _ = feats("val", True)
    if len(f_tr) < 30 or len(f_va) < 10:
        print("Warning: Insufficient corn images for OOD gate; skipping")
        return {}
    classes, means, prec = fit_gate(f_tr, y_tr)
    val_dists = distances(f_va, means, prec)
    threshold = float(np.percentile(val_dists, percentile))

    rep = {"classes": classes, "percentile": percentile, "threshold": threshold, "train_corn_images": len(f_tr)}
    f_te, _ = feats("test", True)
    if len(f_te):
        rep["heldout_corn_wrongly_rejected"] = float((distances(f_te, means, prec) > threshold).mean())
        rep["heldout_corn_images"] = len(f_te)
    neg = [feats(k, False)[0] for k in ("train", "val", "test")]
    f_neg = np.concatenate([n for n in neg if len(n)]) if any(len(n) for n in neg) else np.zeros((0, 1280))
    if len(f_neg):
        rep["non_corn_correctly_rejected"] = float((distances(f_neg, means, prec) > threshold).mean())
        rep["non_corn_images"] = len(f_neg)
    np.savez(out_path, means=means.astype("float32"), precision=prec.astype("float32"),
             threshold=np.float64(threshold), classes=np.array(classes))
    return rep


def print_report(rep):
    print("\n== Calibrated OOD Check Report ==")
    print(f"Fitted on {rep.get('train_corn_images', 0)} corn images, classes {rep.get('classes', [])}, threshold: {rep.get('threshold', 0):.2f}")
    if "heldout_corn_wrongly_rejected" in rep:
        print(f"Held-out real corn leaves rejected: {rep['heldout_corn_wrongly_rejected']:.1%} (target ~1.5%)")
    if "non_corn_correctly_rejected" in rep:
        print(f"Out-of-domain (Sugarcane/Text/Objects) correctly rejected: {rep['non_corn_correctly_rejected']:.1%}")


## 3) Dataset Configuration
Configures sources with **Zero-Auth Direct GitHub Mirror for Sugarcane** (no Kaggle 403 error!), PlantVillage crops, and procedural generation.

In [ ]:
from pathlib import Path
import random, shutil, os, json

BASE = Path("/content")
SRC, DATA, OUT = BASE / "src", BASE / "data", BASE / "output"
SEED = 42
random.seed(SEED)

# Pre-defined paths for PlantVillage
PV = SRC / "plantvillage"
PV_COLOR = PV / "raw" / "color"

# 1. Maize leaf datasets
KAGGLE_CORN = ["smaranjitghose/corn-or-maize-leaf-disease-dataset"]

# 2. Sugarcane: Direct Open GitHub Mirror URL (Zero-Auth; never returns 403!)
DIRECT_SUGARCANE_URL = "https://raw.githubusercontent.com/sabidarrow/Sugarcane-Disease-Dataset/main/sg-diseases%201.0.v1-v1.yolov8.zip"

# Optional: If you have a Kaggle account (kaggle.json), set your key below:
# os.environ["KAGGLE_USERNAME"] = "your_username"
# os.environ["KAGGLE_KEY"] = "your_key"
KAGGLE_SUGARCANE = ["smaranjitghose/sugarcane-leaf-disease-dataset"]

# 3. Non-corn crop samples from PlantVillage
PV_NON_CORN_PER_FOLDER = 45

# 4. Everyday objects and flowers
IMAGENETTE_N = 400            # Everyday objects, tools, furniture, vehicles
FLOWERS_N = 300               # Flowers and blooms

# 5. Procedural & Synthetic Generators
GENERATE_PROCEDURAL_MONOCOT = True   # Generates 250 parallel-veined strap leaf samples
GENERATE_SYNTHETIC_DOCUMENTS = True  # Generates 400 notebook pages, text, handwriting

# 6. Optional custom Google Drive folders
USE_DRIVE = False
MY_CORN_ROOTS = []
MY_NON_CORN_ROOTS = []

In [ ]:
# 4) Dataset Collection & Synthetic Generation Helpers
import re, numpy as np, urllib.request, zipfile, io, random, shutil
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".jfif"}
IMG = 224

def canon(folder, overrides=None):
    """Map a folder name to one of our 5 canonical classes."""
    if overrides and folder in overrides:
        return overrides[folder]
    k = re.sub(r"[^a-z0-9]+", "_", folder.lower()).strip("_")
    tokens = set(k.split("_"))

    # Non-maize / negative keywords
    if tokens & {"not", "non", "other", "others", "negative", "background", "not_corn", "not_leaf", "non_corn", "non_maize"}:
        return "Not_Leaf"

    # Monocot grasses (Sugarcane, Sorghum, Bamboo, Rice, Grass)
    monocots = {"sugarcane", "sugar_cane", "saccharum", "cane", "sorghum", "bamboo", "rice", "paddy", "wheat", "grass", "weed", "weeds", "poaceae"}
    if tokens & monocots:
        return "Not_Leaf"

    # Other plant species from PlantVillage, etc.
    non_corn = {"apple", "banana", "blueberry", "cassava", "cherry", "citrus", "coffee", "cotton", "grape", "mango", "olive", "onion", "orange", "peach", "pepper", "potato", "raspberry", "soybean", "squash", "strawberry", "tea", "tomato"}
    if tokens & non_corn:
        return "Not_Leaf"

    # Flowers & blooms
    flowers = {"flower", "flowers", "rose", "orchid", "sunflower", "tulip", "petal", "bloom", "hibiscus", "dahlia"}
    if tokens & flowers:
        return "Not_Leaf"

    # Documents, handwriting, text
    docs = {"text", "document", "documents", "doc", "paper", "writing", "handwriting", "book", "invoice", "receipt", "notes", "code", "drawing"}
    if tokens & docs:
        return "Not_Leaf"

    # Everyday objects and scenes
    objects = {"object", "objects", "imagenet", "imagenette", "cifar", "coco", "car", "vehicle", "animal", "indoor", "room", "synthetic"}
    if tokens & objects:
        return "Not_Leaf"

    # Corn conditions
    if "rust" in k: return "Common_Rust"
    if "blight" in k: return "Blight"
    if tokens & {"gray", "grey", "cercospora", "spot"}: return "Gray_Leaf_Spot"
    if "healthy" in k and ("corn" in k or "maize" in k or len(tokens) == 1): return "Healthy"
    return None

def add_images(root, prefix, force_class=None, cap=None, folder_filter=None):
    root, copied = Path(root), 0
    dirs = [root] + sorted(d for d in root.rglob("*") if d.is_dir())
    for d in dirs:
        if folder_filter and not folder_filter(d.name): continue
        files = [f for f in d.iterdir() if f.is_file() and f.suffix.lower() in EXTS]
        if not files: continue
        cls = force_class or canon(d.name)
        if cls is None: continue
        random.shuffle(files)
        if cap: files = files[:cap]
        out = DATA / cls; out.mkdir(parents=True, exist_ok=True)
        for f in files:
            shutil.copy2(f, out / f"{prefix}__{re.sub(r'[^A-Za-z0-9]+', '', d.name)[:20]}__{f.name}")
        copied += len(files)
    return copied

def download_direct_sugarcane(dest_dir, max_images=500):
    """Downloads real sugarcane leaf images from public GitHub archive with zero authentication."""
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    try:
        print("Downloading real Sugarcane leaf dataset from open GitHub mirror (zero-auth)... ")
        req = urllib.request.Request(DIRECT_SUGARCANE_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=45) as resp:
            data = resp.read()
        zf = zipfile.ZipFile(io.BytesIO(data))
        img_members = [m for m in zf.namelist() if m.lower().endswith((".jpg", ".jpeg", ".png")) and not m.startswith("__")]
        random.shuffle(img_members)
        count = 0
        for m in img_members[:max_images]:
            img_data = zf.read(m)
            fname = Path(m).name
            (dest / f"sugarcane_real_{count:04d}_{fname}").write_bytes(img_data)
            count += 1
        print(f"--> Successfully downloaded and extracted {count} real Sugarcane leaf images!")
        return count
    except Exception as e:
        print(f"Direct sugarcane download notice: {e}")
        return 0

def kaggle_get(slug):
    import kagglehub
    try:
        path = kagglehub.dataset_download(slug)
        print("Successfully downloaded", slug, "->", path)
        return Path(path)
    except Exception as e:
        print("Note: Kagglehub skipped", slug, "(Kaggle auth required). Using public zero-auth mirrors.")
        return None

def dump_tfds(name, split, n, class_dir, prefix):
    import tensorflow_datasets as tfds
    try:
        ds = tfds.load(name, split=split, shuffle_files=True).shuffle(2000, seed=SEED).take(n)
        out = DATA / class_dir; out.mkdir(parents=True, exist_ok=True)
        k = 0
        for ex in tfds.as_numpy(ds):
            Image.fromarray(ex["image"]).convert("RGB").save(out / f"{prefix}__{k}.jpg", quality=92); k += 1
        print(f"Saved {k} images from TFDS {name}"); return k
    except Exception as e:
        print("SKIPPED TFDS", name, "->", str(e)[:150]); return 0

def make_procedural_monocot_leaves(n, class_dir, prefix):
    """Generates procedural strap leaves with parallel venation and central midrib."""
    out = DATA / class_dir; out.mkdir(parents=True, exist_ok=True)
    for i in range(n):
        base_green = (random.randint(40, 90), random.randint(110, 180), random.randint(30, 70))
        bg = (random.randint(200, 240), random.randint(190, 230), random.randint(180, 210))
        img = Image.new("RGB", (IMG, IMG), bg)
        angle = random.uniform(-60, 60)
        blade = Image.new("RGBA", (IMG * 2, IMG * 2), (0, 0, 0, 0))
        bdraw = ImageDraw.Draw(blade)
        cx = IMG; w = random.randint(28, 55)
        bdraw.rectangle([(cx - w, 0), (cx + w, IMG * 2)], fill=(*base_green, 255))
        for vx in range(cx - w + 2, cx + w - 2, 3):
            v_green = (base_green[0] + random.randint(-15, 15), base_green[1] + random.randint(-15, 15), base_green[2] + random.randint(-10, 10))
            bdraw.line([(vx, 0), (vx, IMG * 2)], fill=(*v_green, 220), width=1)
        bdraw.line([(cx, 0), (cx, IMG * 2)], fill=(210, 230, 180, 255), width=random.randint(2, 4))
        for _ in range(random.randint(2, 8)):
            lx, ly = random.randint(cx - w + 4, cx + w - 4), random.randint(20, IMG * 2 - 20)
            bdraw.ellipse([(lx - 3, ly - 7), (lx + 3, ly + 7)], fill=(random.randint(120, 160), random.randint(60, 90), random.randint(20, 40), 220))
        blade_rot = blade.rotate(angle, resample=Image.Resampling.BILINEAR)
        img.paste(blade_rot, (-IMG // 2, -IMG // 2), blade_rot)
        img.save(out / f"{prefix}__{i:04d}.jpg", quality=92)
    print(f"Generated {n} procedural monocot/grass leaf images with parallel venation.")

def make_synthetic_documents(n, class_dir, prefix):
    out = DATA / class_dir; out.mkdir(parents=True, exist_ok=True)
    font = ImageFont.load_default()
    texts = [
        "Leafcheck Agricultural Report\nSample #9042\nDiagnosis: Pending\nVegetation Status: Analyzed\nVerified by: Agronomist",
        "Field Inspection Notes:\nPlot A: North sector\nCrop: Field examination\nConditions: Sunny, 28C\nHumidity: 65%",
        "INVOICE #4412\nDate: 2026-10-24\nItem: Potassium Fertilizer\nQuantity: 10 units\nTotal: $350.00\nStatus: Paid",
        "Linear Algebra & Vision:\nMatrix eigenvalues:\nlambda = det(A - lambda*I)\nRank deficient check: True",
    ]
    for i in range(n):
        bg_choice = i % 4
        bg = (255, 255, 255) if bg_choice == 0 else ((250, 245, 210) if bg_choice == 1 else (235, 235, 235))
        img = Image.new("RGB", (IMG, IMG), bg)
        draw = ImageDraw.Draw(img)
        if bg_choice == 1:
            for y in range(25, IMG, 20): draw.line([(0, y), (IMG, y)], fill=(210, 190, 160))
            draw.line([(35, 0), (35, IMG)], fill=(230, 140, 140), width=1)
        if i % 2 == 0:
            draw.multiline_text((40, 30), random.choice(texts), fill=(20, 20, 20), font=font, spacing=4)
        else:
            for line_idx in range(5):
                y_base = 35 + line_idx * 35
                pts = [(x, y_base + random.randint(-5, 5)) for x in range(25, IMG - 25, 12)]
                draw.line(pts, fill=(15, 25, 120), width=random.choice([1, 2]))
        img.save(out / f"{prefix}__{i:04d}.jpg", quality=92)
    print(f"Generated {n} synthetic document and handwriting images.")

def count_classes():
    return {d.name: len([f for f in d.iterdir() if f.suffix.lower() in EXTS]) for d in sorted(DATA.iterdir()) if d.is_dir()} if DATA.exists() else {}

## 5) Download & Assemble Dataset

In [ ]:
# 5a) Clone PlantVillage from GitHub
SRC.mkdir(parents=True, exist_ok=True)
if not PV.exists():
    !git clone -q --depth 1 --filter=blob:none --sparse https://github.com/spMohanty/PlantVillage-Dataset.git {PV}
    !cd {PV} && git sparse-checkout set raw/color
print("PlantVillage available:", PV_COLOR.exists())

In [ ]:
# 5b) Assemble MAIZE images -> Blight, Common_Rust, Gray_Leaf_Spot, Healthy
shutil.rmtree(DATA, ignore_errors=True); DATA.mkdir(parents=True)

if PV_COLOR.exists():
    n = add_images(PV_COLOR, "pv_corn", folder_filter=lambda name: name.lower().startswith("corn"))
    print("PlantVillage maize images copied:", n)

for slug in KAGGLE_CORN:
    p = kaggle_get(slug)
    if p: print(f"  {slug}: copied", add_images(p, "kaggle_corn"))

if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    for r in MY_CORN_ROOTS: print("  my maize photos:", r, add_images(r, "mine"))

print(count_classes())

In [ ]:
# 5c) Assemble NON-MAIZE images -> Not_Leaf
# 1. Real Sugarcane leaves (Zero-Auth Open GitHub dataset mirror)
n_sc = download_direct_sugarcane(DATA / "Not_Leaf", max_images=500)

# Optional Kaggle fallback if user has credentials
if n_sc < 50:
    for slug in KAGGLE_SUGARCANE:
        p = kaggle_get(slug)
        if p:
            n_sc = add_images(p, "sugarcane", force_class="Not_Leaf", cap=500)
            if n_sc > 0: break

# 2. Procedural Monocot Grass & Sugarcane Leaves (guaranteed parallel-venation negatives)
if GENERATE_PROCEDURAL_MONOCOT:
    make_procedural_monocot_leaves(250, "Not_Leaf", "monocot_grass")

# 3. Other plant crops from PlantVillage
if PV_COLOR.exists():
    n_other = add_images(PV_COLOR, "pv_other", force_class="Not_Leaf", cap=PV_NON_CORN_PER_FOLDER,
                         folder_filter=lambda name: not name.lower().startswith("corn"))
    print("Other crop leaves copied:", n_other)

# 4. Synthetic documents, handwriting, and text
if GENERATE_SYNTHETIC_DOCUMENTS:
    make_synthetic_documents(400, "Not_Leaf", "doc")

# 5. Everyday objects and flowers via TFDS
dump_tfds("imagenette/320px-v2", "train", IMAGENETTE_N, "Not_Leaf", "imagenette")
dump_tfds("oxford_flowers102", "train", FLOWERS_N, "Not_Leaf", "flowers")

if USE_DRIVE:
    for r in MY_NON_CORN_ROOTS: print("  my non-corn photos:", r, add_images(r, "mine_neg", force_class="Not_Leaf"))

print("Final class counts:", count_classes())

## 6) Verify Dataset Before Training

In [ ]:
counts = count_classes()
for k, v in counts.items(): print(f"{k:18s} {v:6d}")
need = {"Blight", "Common_Rust", "Gray_Leaf_Spot", "Healthy", "Not_Leaf"}
missing = need - set(counts)
assert not missing, f"Missing classes: {missing}. Please re-run the download cells above."
assert min(counts.values()) >= 50, "Each class must have at least 50 images."
print("\nAll 5 classes verified! Ready for training.")

## 7) Train (Cleans, Deduplicates, Trains MobileNetV2, and Calibrates OOD Check)

In [ ]:
!python train.py --data {DATA} --out {OUT} --max-per-class 1500 --workers 4

## 8) Evaluation & Confusion Matrix

In [ ]:
import json
from pathlib import Path
m = json.load(open(OUT / "metrics.json"))
print("Test Accuracy :", round(m["test_accuracy"], 4))
print("Test Macro-F1 :", round(m["test_macro_f1"], 4))
print("Class counts  :", m["class_counts"])
for c, r in m["per_class"].items():
    print(f"{c:16s} precision {r['precision']:.3f}  recall {r['recall']:.3f}  f1 {r['f1-score']:.3f}")
from IPython.display import Image as show
show(str(OUT / "confusion_matrix.png"))

## 9) Package & Download Model for Local App
Downloads `leafcheck_model.zip`. Extract into `backend/model/` and restart your backend!

In [ ]:
import shutil
from pathlib import Path
deploy = BASE / "deploy"; shutil.rmtree(deploy, ignore_errors=True); deploy.mkdir()
for f in ["corn_leaf_disease_model_v3.keras", "class_names.json", "ood_stats.npz", "metrics.json", "confusion_matrix.png"]:
    shutil.copy2(OUT / f, deploy / f)
shutil.make_archive(str(BASE / "leafcheck_model"), "zip", deploy)
from google.colab import files
files.download(str(BASE / "leafcheck_model.zip"))
print("\nDone! Unzip leafcheck_model.zip into d:/final-year-project/backend/model/ and restart uvicorn.")